## tl;dr

Al 28 de agosto de 2026 no hay alumnos activos que puedan iniciar la acreditación de forma segura inmediatamente. Hay 18 académicamente listos con informes pendientes; 4 están a una sola corrección. La evidencia documental nueva cubre 0 de sus 114 PPS, por lo que hoy los 18 caerían al circuito asistido después de completar sus informes.

## Context & Methods

Unidad de análisis: alumno activo, rol alumno, sin solicitud previa en `finalizacion_pps`. Las reglas replican la función de transición con una corrección conservadora: un `informe_estado` nulo cuenta como pendiente. Requisitos: 250 horas totales, 70 horas en la orientación elegida, 3 orientaciones y ninguna PPS activa.

### Key Assumptions

- Las PPS desaprobadas, canceladas, abandonadas o no concretadas quedan excluidas.
- Una PPS online no requiere planilla de asistencia.
- Una PPS presencial sólo es automática con evidencia `detected` o `assumed` y confianza de al menos 0,900.
- Fuente viva: Supabase, tablas `estudiantes`, `practicas`, `finalizacion_pps` y `moodle_grade_snapshots`.

## Data

Snapshot agregado obtenido por consultas SQL de sólo lectura el 28 de agosto de 2026 a las 08:21 ART. No contiene nombres, legajos, correos ni identificadores de alumnos.

In [1]:
snapshot = {
    'active_without_finalization': 225,
    'with_eligible_practices': 212,
    'without_eligible_practices': 13,
    'automatic_now': 0,
    'assisted_now_safe': 0,
    'academic_ready_reports_pending': 18,
    'reports_complete_academic_gaps': 21,
    'reports_and_academic_gaps': 173,
    'one_report_away': 4,
    'attendance_uncertain_one_report_away': 21,
    'attendance_uncertain_academic_ready': 92,
    'classified_candidate_practices': 0,
    'all_snapshots': 726,
    'classified_snapshots': 10,
    'current_evaluator_false_positives': 6,
}
snapshot

{'active_without_finalization': 225,
 'with_eligible_practices': 212,
 'without_eligible_practices': 13,
 'automatic_now': 0,
 'assisted_now_safe': 0,
 'academic_ready_reports_pending': 18,
 'reports_complete_academic_gaps': 21,
 'reports_and_academic_gaps': 173,
 'one_report_away': 4,
 'attendance_uncertain_one_report_away': 21,
 'attendance_uncertain_academic_ready': 92,
 'classified_candidate_practices': 0,
 'all_snapshots': 726,
 'classified_snapshots': 10,
 'current_evaluator_false_positives': 6}

## Results

In [2]:
pipeline_total = (
    snapshot['automatic_now']
    + snapshot['assisted_now_safe']
    + snapshot['academic_ready_reports_pending']
    + snapshot['reports_complete_academic_gaps']
    + snapshot['reports_and_academic_gaps']
)
classifier_coverage = snapshot['classified_snapshots'] / snapshot['all_snapshots']
assert pipeline_total == snapshot['with_eligible_practices']
assert snapshot['automatic_now'] + snapshot['assisted_now_safe'] == 0
print(f'Alumnos reconciliados: {pipeline_total}')
print(f'Cobertura del clasificador: {classifier_coverage:.2%}')
print(f'A una corrección: {snapshot["one_report_away"]}')
print(f'Potencial académicamente listo: {snapshot["academic_ready_reports_pending"]}')

Alumnos reconciliados: 212
Cobertura del clasificador: 1.38%
A una corrección: 4
Potencial académicamente listo: 18


## Takeaways

- Resultado seguro inmediato: 0 automáticos y 0 asistidos.
- Cohorte más próxima: 4 alumnos a una corrección; con la evidencia actual los 4 serían asistidos.
- Potencial total: 18 alumnos académicamente listos, pero todavía con informes pendientes.
- La estimación automática no es estable hasta instalar el puente nuevo: sólo 10 de 726 snapshots tienen clasificación y ninguno pertenece a las 114 PPS de esos 18 alumnos.
- Antes de activar hay que corregir el uso de `bool_and` para que los estados nulos no se interpreten como informes completos.